In [1]:
%load_ext autoreload
%autoreload 2

# inverse the schro
# constrain: no obs only prediction
# now the prediction is the integrated belief, the preception.


In [2]:
import pickle
import torch
import ray
import numpy as np
from timeit import default_timer as timer
from pathlib import Path
import copy
import sys
sys.path.append('..')

from cmaes import CMA
import copy
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')



from torch.distributions.multivariate_normal import MultivariateNormal
from matplotlib import pyplot as plt
import time
from stable_baselines3 import TD3
torch.manual_seed(42)
from numpy import pi
from InverseFuncs import *
from firefly_task import ffacc_real
from env_config import Config
import ray

arg = Config()
import os
from timeit import default_timer as timer
from plot_ult import process_inv, run_trial
from notification import notify
from monkey_functions import *
import pandas as pd

import configparser
config = configparser.ConfigParser()
config.read_file(open('privateconfig'))
resdir = Path(config['Datafolder']['data'])
workdir = Path(config['Codefolder']['workspace'])
os.chdir(workdir)


In [3]:
resdir, workdir

(PosixPath('/Users/yc/Documents/lab_data'), PosixPath('/Users/yc/repo/irc'))

In [4]:
datafiles=list(resdir.rglob('schro_normal/packed'))
datafiles

[PosixPath('/Users/yc/Documents/lab_data/schro_normal/packed')]

In [5]:
# create the dataset
states, actions, tasks=[],[],[]
for file in datafiles:
    # Use pandas' built-in pickle reader instead
    df = pd.read_pickle(file)
    df = datawash(df) # remmove short trials
    df = df[df.category=='normal'] # only take normal trials, not skip and wrong tar, carzy and lazy.
    df=df[df.target_r>250]
    s, a, t=monkey_data_downsampled(df,factor=0.0025)
    states+=s
    actions+=a
    tasks+=t



In [6]:
df

,gain_v,gain_w,perturb_vpeakmax,perturb_wpeakmax,perturb_sigma,perturb_dur,perturb_vpeak,perturb_wpeak,perturb_start_time,perturb_start_time_ori,...,rewarded,relative_radius,relative_angle,time,trial_dur,action_v,action_w,relative_radius_end,relative_angle_end,category
4,200.0,90.0,200.0,200.0,0.2,1.0,0.0,-0.0,NaN,0.0,...,True,"[255.9574437436268, 239.53399721868743, 230.02...","[160.89166954871712, 161.34341228343865, 163.3...","[0.0, 0.09999999999999906, 0.19999999999999932...",1.6560,"[0.9605719757080078, 0.7621059926350912, 0.087...","[0.061517090267605254, 0.31643269150345416, 0....",59.638517,179.716260,normal
8,200.0,90.0,200.0,200.0,0.2,1.0,0.0,-0.0,NaN,0.0,...,False,"[388.43870978604355, 385.61508179234505, 376.9...","[163.7042555267681, 163.58102610723114, 163.91...","[0.0, 0.09999999999999906, 0.19999999999999812...",1.7340,"[0.03973454713821411, 0.3096320152282715, 0.63...","[-4.025449355443319e-05, -7.30326330220239e-05...",139.729670,-125.994694,normal
10,200.0,90.0,200.0,200.0,0.2,1.0,0.0,-0.0,NaN,0.0,...,False,"[263.99168384003417, 249.56256540888614, 234.3...","[-173.9885804062289, -175.15163239098305, -176...","[0.0, 0.10000000000000143, 0.20000000000000287...",1.4016,"[0.5953985977172852, 0.7621466827392578, 0.762...","[-0.15688365300496418, -0.16465207205878363, -...",76.539287,170.463775,normal
11,200.0,90.0,200.0,200.0,0.2,1.0,0.0,-0.0,NaN,0.0,...,False,"[276.34426216060945, 272.91047097784156, 260.7...","[-170.53891602189657, -170.42497499883993, -17...","[0.0, 0.10000000000000143, 0.20000000000000048...",1.4112,"[0.03186101198196411, 0.43656477610270183, 0.7...","[-4.025449355443319e-05, -0.04553631323355217,...",81.268930,-172.071968,normal
12,200.0,90.0,200.0,200.0,0.2,1.0,0.0,-0.0,NaN,0.0,...,True,"[365.12966058228295, 357.8235605608614, 345.24...","[-150.13742263702312, -150.3483044573029, -150...","[0.0, 0.09999999999999669, 0.19999999999999576...",2.3172,"[0.2540900230407715, 0.5714931615193686, 0.785...","[-0.05451752344767253, -0.12554775167394566, -...",43.904835,-76.085950,normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1477,200.0,90.0,200.0,200.0,0.2,1.0,0.0,-0.0,NaN,0.0,...,True,"[331.5239201433473, 326.5503130232327, 309.994...","[-169.35531042224906, -169.25469723369594, -16...","[0.0, 0.09999999999960589, 0.19999999999966653...",1.6260,"[0.03182767152786255, 0.6511514282226563, 0.99...","[-0.00039522647857666017, -0.00059189531538221...",58.619694,160.261048,normal
1480,200.0,90.0,200.0,200.0,0.2,1.0,0.0,-0.0,NaN,0.0,...,False,"[287.5424732147693, 284.9757000660227, 277.397...","[-163.65458030097437, -163.50626181029503, -16...","[0.0, 0.10000000000006064, 0.20000000000012128...",1.7436,"[0.03988431215286255, 0.2861611747741699, 0.59...","[-0.00029689206017388236, -6.744508390073609e-...",73.363784,-165.469064,normal
1481,200.0,90.0,200.0,200.0,0.2,1.0,0.0,-0.0,NaN,0.0,...,True,"[287.7684839112066, 267.9265879018531, 248.081...","[178.01538998929695, 178.5296101370951, 178.96...","[0.0, 0.10000000000006064, 0.20000000000012128...",1.2828,"[0.9921548461914063, 0.9924396769205729, 0.992...","[0.09980754852294922, 0.0613915690669307, 0.05...",52.939390,169.177691,normal
1482,200.0,90.0,200.0,200.0,0.2,1.0,0.0,-0.0,NaN,0.0,...,True,"[294.8402794882415, 287.1152583125223, 271.500...","[-168.92023858105657, -169.42854761248012, -17...","[0.0, 0.10000000000006064, 0.20000000000012128...",1.8192,"[0.05569242000579834, 0.7622354125976563, 0.81...","[-1.8888049655490452e-06, -0.1731687969631619,...",41.549533,-150.517489,normal


In [7]:
# load agent and task
env=ffacc_real.FireFlyPaper2(arg)
env.episode_len=50
phi=torch.tensor([[0.5],
            [pi/2],
            [0.001],
            [0.001],
            [0.001],
            [0.001],
            [0.13],
            [0.001],
            [0.001],
            [0.001],
            [0.001],
    ])
agent_=TD3.load('trained_agent/paper.zip')
agent=agent_.actor.mu.cpu()


In [8]:
del df
print(len(states))

4702


In [ ]:
resfile = Path('res/shro_0430')

# decide if to continue
optimizer = None
log = []
if resfile.is_file():
    print('continue on previous inverse...')
    with open(resfile, 'rb') as f:
        log = pickle.load(f)
    optimizer = log[-1][0]
else:
    print('starting new inverse ...')

# use localhost
ray.init(log_to_driver=False, ignore_reinit_error=True)

# Define which parameters are fixed (True = fixed, False = optimized)
# We're fixing only the goal_radius (index 6)
fixed_mask = np.array([
    False,  # _prov_gains
    False,  # _prow_gains
    False,  # _prov_noise_stds
    False,  # _prow_noise_stds
    False,  # _obsv_noise_stds
    False,  # _obsw_noise_stds
    True,   # _goal_radius - fixed at 0.13
    False,  # _dev_v_cost_factor
    False,  # _dev_w_cost_factor
    False,  # _inital_x_std
    False   # _inital_y_std
])

# Fixed parameter values - only goal_radius is fixed
fixed_values = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.13, 0.0, 0.0, 0.0, 0.0], dtype=np.float32)

# Function to map reduced parameters to full parameter set
def map_to_full_params(x_reduced):
    x_full = fixed_values.copy()
    x_full[~fixed_mask] = x_reduced
    return x_full

# Function to extract reduced parameters from full set
def extract_reduced_params(x_full):
    return x_full[~fixed_mask]

@ray.remote
def getlogll(x_reduced):
    # Map reduced parameters to full parameter set
    x_full = map_to_full_params(x_reduced)
    
    with torch.no_grad():
        return monkeyloss_(agent, actions, tasks, phi, 
                          torch.tensor(x_full).t(), env, 
                          action_var=1e-3, num_iteration=1, 
                          states=states, samples=5, gpu=False).item()

if not optimizer:
    # Define full bounds
    lower_bounds_full = np.array([0.2, 0.7, 0.1, 0.49, 0.1, 0.49, 0.129, 0.1, 0.1, 0.1, 0.1], dtype=np.float32)
    upper_bounds_full = np.array([1.0, 2.0, 2.0, 0.5, 2.0, 0.5, 0.131, 0.9, 0.9, 0.9, 0.6], dtype=np.float32)
    
    # Extract bounds for only the parameters we're optimizing
    lower_bounds = lower_bounds_full[~fixed_mask]
    upper_bounds = upper_bounds_full[~fixed_mask]
    
    # Init condition for full parameter set
    init_theta_full = torch.tensor([[0.5], [1.0], [0.5], [0.5], [0.5], 
                                   [0.5], [0.13], [0.5], [0.5], [0.5], [0.5]])
    
    # Extract initial values for only parameters we're optimizing
    init_theta = init_theta_full.view(-1).numpy()[~fixed_mask]
    
    # Determine dimensionality of optimized parameter space
    dim = len(init_theta)
    
    # Calculate population size using rule of thumb
    population_size = 4 + int(3 * np.log(dim))
    
    # Adaptive sigma based on parameter ranges
    param_ranges = upper_bounds - lower_bounds
    adaptive_sigma = np.mean(param_ranges) / 4
    
    # Create CMA optimizer with improved parameters (optimizing only non-fixed parameters)
    optimizer = CMA(
        mean=init_theta,
        sigma=adaptive_sigma,
        population_size=population_size,
        seed=42,  # For reproducibility
    )
    
    # Set bounds for optimized parameters
    optimizer.set_bounds(np.vstack([lower_bounds, upper_bounds]).transpose())

# Optimization loop
for generation in range(len(log), len(log) + 399):
    start = timer()
    
    # Generate and evaluate solutions
    xs_reduced = []
    for _ in range(optimizer.population_size):
        x = optimizer.ask().astype('float32')
        xs_reduced.append(x)
    
    # Evaluate solutions
    solution_values = ray.get([getlogll.remote(p) for p in xs_reduced])
    
    # Calculate mean log-likelihood
    meanlogll = np.mean(solution_values)
    
    # Pair solutions with their values
    solutions = [[x, s] for x, s in zip(xs_reduced, solution_values)]
    
    # Update optimizer with results
    optimizer.tell(solutions)
    
    # Convert reduced parameter sets to full parameter sets for logging and display
    xs_full = [map_to_full_params(x) for x in xs_reduced]
    
    # Store results in log
    log.append([copy.deepcopy(optimizer), xs_full, solution_values])
    
    # Save progress
    with open(resfile, 'wb+') as handle:
        pickle.dump(log, handle, protocol=pickle.HIGHEST_PROTOCOL)
    
    # Display progress
    print('done, ', timer() - start)
    print("generation: ", generation, '-logll: ', meanlogll)
    
    # Get current mean in full parameter space
    full_mean = map_to_full_params(optimizer._mean)
    print('cur estimation ', ["{0:0.2f}".format(i) for i in full_mean])
    
    # Get uncertainty for optimized parameters
    uncertainty = np.diag(optimizer._C)**0.5
    
    # Create full uncertainty vector (fixed parameters have zero uncertainty)
    full_uncertainty = np.zeros(len(fixed_mask))
    full_uncertainty[~fixed_mask] = uncertainty
    print('cur uncertainty ', ["{0:0.2f}".format(i) for i in full_uncertainty])
    
    # Send notification with current estimate
    notify(msg="".join(['{:0.1f} '.format(i) for i in full_mean]))
    
    # Check stopping criteria
    if optimizer.should_stop():
        print('stop at {}th generation'.format(str(generation)))
        break

# After optimization, get the best solution in full parameter space
best_params_full = map_to_full_params(optimizer._mean)
print("Final optimized parameters:")
param_names = ['_prov_gains', '_prow_gains', '_prov_noise_stds', '_prow_noise_stds',
               '_obsv_noise_stds', '_obsw_noise_stds', '_goal_radius', '_dev_v_cost_factor',
               '_dev_w_cost_factor', '_inital_x_std', '_inital_y_std']

for name, value, is_fixed in zip(param_names, best_params_full, fixed_mask):
    status = "FIXED" if is_fixed else "optimized"
    print(f"{name}: {value:.4f} ({status})")

starting new inverse ...


2025-04-30 14:37:17,842	INFO worker.py:1788 -- Started a local Ray instance.


done,  689.1628015419992
generation:  0 -logll:  47.98752098083496
cur estimation  ['0.42', '1.09', '0.36', '0.49', '0.65', '0.49', '0.13', '0.69', '0.61', '0.52', '0.35']
cur uncertainty  ['0.97', '1.00', '1.00', '1.00', '0.99', '1.00', '0.00', '1.02', '1.02', '0.96', '0.99']
done,  695.858318500017
generation:  1 -logll:  43.54183387756348
cur estimation  ['0.41', '1.20', '0.32', '0.49', '0.68', '0.50', '0.13', '0.54', '0.42', '0.56', '0.38']
cur uncertainty  ['0.93', '1.00', '0.98', '1.00', '0.98', '1.00', '0.00', '1.01', '1.03', '0.95', '0.96']
done,  591.7182646250003
generation:  2 -logll:  41.09482307434082
cur estimation  ['0.49', '1.33', '0.47', '0.50', '0.84', '0.50', '0.13', '0.38', '0.34', '0.64', '0.40']
cur uncertainty  ['0.92', '0.99', '0.98', '1.00', '1.00', '1.00', '0.00', '1.01', '1.01', '0.95', '0.95']
done,  624.2444130000076
generation:  3 -logll:  39.31506004333496
cur estimation  ['0.46', '1.44', '0.64', '0.49', '0.81', '0.50', '0.13', '0.38', '0.35', '0.51', '0.